In [ ]:
# Load diabetic_data.csv and perform descriptive stats on numerical features
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# Read the CSV
file_path = 'diabetic_data.csv'
diabetic_df = pd.read_csv(file_path, encoding='ascii')

# Identify numeric columns (coerce errors later if needed)
numeric_cols = diabetic_df.select_dtypes(include=['number']).columns.tolist()

# Basic descriptive statistics for numeric columns
describe_numeric = diabetic_df[numeric_cols].describe()

print(diabetic_df.head())
print(describe_numeric)

# Visualize distribution of categorical features: race and gender

sns.set(style='whitegrid')

# Barchart for race
plt.figure(figsize=(8,4))
ax1 = sns.countplot(data=diabetic_df, x='race', order=diabetic_df['race'].value_counts().index)
plt.title('Distribution of Race')
plt.xlabel('Race')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Barchart for gender
plt.figure(figsize=(5,4))
ax2 = sns.countplot(data=diabetic_df, x='gender', order=diabetic_df['gender'].value_counts().index)
plt.title('Distribution of Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Also show value counts as basic tables
print(diabetic_df['race'].value_counts(dropna=False))
print(diabetic_df['gender'].value_counts(dropna=False))

# Ensure seaborn style
sns.set(style='whitegrid')

# Focus on age and readmitted columns
age_readmit_ct = pd.crosstab(diabetic_df['age'], diabetic_df['readmitted'])
age_readmit_prop = age_readmit_ct.div(age_readmit_ct.sum(axis=1), axis=0)

print(age_readmit_ct.head())
print(age_readmit_prop.head())

# Stacked bar of proportions by age group
plt.figure(figsize=(10,6))
age_readmit_prop.loc[sorted(age_readmit_prop.index)].plot(kind='bar', stacked=True, figsize=(10,6))
plt.title('Proportion of Readmission Status by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Proportion')
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

# Select only numerical columns
numeric_df = diabetic_df.select_dtypes(include='number')

# Compute correlation matrix
corr_matrix = numeric_df.corr()

print("Correlation Matrix:")
print(corr_matrix)

# Visualize correlation matrix as a heatmap
plt.figure(figsize=(12, 8))
plt.imshow(corr_matrix, interpolation='nearest')
plt.title("Correlation Heatmap")
plt.colorbar()
plt.xticks(range(len(numeric_df.columns)), numeric_df.columns, rotation=90)
plt.yticks(range(len(numeric_df.columns)), numeric_df.columns)
plt.show()

# # Identify medication columns
#
med_change_cols = [col for col in diabetic_df.columns if "change" in col.lower()]
med_cols = [col for col in diabetic_df.columns if "med" in col.lower() and col not in med_change_cols]
#
# print("Medication change columns:", med_change_cols)
# print("Medication columns:", med_cols)

# Count total medications (anything not No/None)

def count_meds(row):
    count = 0
    for col in med_cols:
        val = str(row[col]).strip().lower()
        if val not in ['no', 'none', 'nan', '']:
            count += 1
    return count

diabetic_df['total_medications'] = diabetic_df.apply(count_meds, axis=1)

# Count medication changes

med_change_counts = {}
for col in med_change_cols:
    med_change_counts[col] = (diabetic_df[col].astype(str).str.lower() != 'no').sum()

med_change_counts = pd.Series(med_change_counts)
print("\nMedication change counts:")
print(med_change_counts)

# Distribution of total medications taken

total_med_dist = diabetic_df['total_medications'].value_counts().sort_index()
print("\nDistribution of total medications taken:")
print(total_med_dist)

# Barplot:Total medications distribution
colors = plt.cm.tab20(np.linspace(0, 1, len(total_med_dist)))

plt.figure(figsize=(10, 6))
plt.bar(total_med_dist.index, total_med_dist.values, color=colors)

plt.title("Distribution of Total Medications Taken")
plt.xlabel("Number of Medications")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Diagnosis columns
diag_cols = ["diag_1", "diag_2", "diag_3"]

# Convert diagnosis codes to numeric when possible
for col in diag_cols:
    diabetic_df[col] = pd.to_numeric(diabetic_df[col], errors='coerce')

# Function to map ICD-9 codes to major categories
def map_icd_category(code):
    if pd.isna(code):
        return "Unknown"
    code = float(code)

    if 390 <= code <= 459 or code == 785:
        return "Circulatory"
    elif 460 <= code <= 519 or code == 786:
        return "Respiratory"
    elif 520 <= code <= 579 or code == 787:
        return "Digestive"
    elif 250 <= code <= 250.99:
        return "Diabetes"
    elif 800 <= code <= 999:
        return "Injury"
    elif 710 <= code <= 739:
        return "Musculoskeletal"
    elif 580 <= code <= 629 or code == 788:
        return "Genitourinary"
    elif 140 <= code <= 239:
        return "Neoplasms"
    else:
        return "Other"

# Apply mapping
for col in diag_cols:
    diabetic_df[col + "_cat"] = diabetic_df[col].apply(map_icd_category)

# Combine and count frequencies
all_diagnoses = pd.concat([diabetic_df[col + "_cat"] for col in diag_cols])
diagnosis_dist = all_diagnoses.value_counts()

print("\nDiagnosis Category Distribution:")
print(diagnosis_dist)

# Plot distribution
plt.figure(figsize=(12, 6))
plt.bar(diagnosis_dist.index, diagnosis_dist.values)
plt.title("Distribution of Diagnosis Categories")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Admission Type distribution
adm_type_counts = diabetic_df['admission_type_id'].value_counts()

# Admission Source distribution
adm_source_counts = diabetic_df['admission_source_id'].value_counts()

# Discharge Disposition distribution
discharge_counts = diabetic_df['discharge_disposition_id'].value_counts()

# Barplot: Admission Type Distribution
plt.figure(figsize=(10, 6))
plt.barh(adm_type_counts.index.astype(str), adm_type_counts.values, color="skyblue")
plt.title("Distribution of Admission Types")
plt.xlabel("Count")
plt.ylabel("Admission Type ID")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Barplot: Admission Source Distribution
plt.figure(figsize=(10, 6))
plt.barh(adm_source_counts.index.astype(str), adm_source_counts.values, color="salmon")
plt.title("Distribution of Admission Sources")
plt.xlabel("Count")
plt.ylabel("Admission Source ID")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Barplot: Discharge Disposition Distribution
plt.figure(figsize=(10, 6))
plt.barh(discharge_counts.index.astype(str), discharge_counts.values, color="mediumseagreen")
plt.title("Distribution of Discharge Dispositions")
plt.xlabel("Count")
plt.ylabel("Discharge Disposition ID")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Select Numerical Columns
num_cols = diabetic_df.select_dtypes(include=['int64', 'float64']).columns

# Detect Outliers Using IQR Method
outlier_summary = {}

for col in num_cols:
    Q1 = diabetic_df[col].quantile(0.25)
    Q3 = diabetic_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = diabetic_df[(diabetic_df[col] < lower_bound) | (diabetic_df[col] > upper_bound)][col]
    outlier_summary[col] = len(outliers)

# Visualize Outliers Using Boxplots
plt.figure(figsize=(14, 8))

diabetic_df[num_cols].boxplot()
plt.title("Boxplot of Numerical Features (Outlier Detection)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Histogram : Distribution
diabetic_df[num_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()





































































